<a href="https://colab.research.google.com/github/yuniecorn-dev/Numerai_ESAA/blob/main/260701_%EB%B0%A9%ED%95%99_%EA%B3%BC%EC%A0%9C_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install numerapi pandas pyarrow lightgbm

In [4]:
from numerapi import NumerAPI

napi = NumerAPI()
napi.download_dataset("v5.2/train.parquet", "train.parquet")
napi.download_dataset("v5.2/validation.parquet", "validation.parquet")
napi.download_dataset("v5.2/features.json", "features.json")

train.parquet: 100%|██████████| 2.60G/2.60G [01:45<00:00, 24.6MB/s]
validation.parquet: 100%|██████████| 4.32G/4.32G [02:58<00:00, 24.2MB/s]
features.json: 100%|██████████| 323k/323k [00:00<00:00, 1.06MB/s]


'features.json'

In [5]:
import pandas as pd

train = pd.read_parquet("train.parquet")
print(train.shape)
print(train.head())

(2746268, 2791)
                   era data_type  feature_shaded_hallucinatory_dactylology  \
id                                                                           
n0007b5abb0c3a25  0001     train                                         3   
n003bba8a98662e4  0001     train                                         4   
n003bee128c2fcfc  0001     train                                         2   
n0048ac83aff7194  0001     train                                         2   
n0055a2401ba6480  0001     train                                         4   

                  feature_itinerant_hexahedral_photoengraver  \
id                                                             
n0007b5abb0c3a25                                           4   
n003bba8a98662e4                                           2   
n003bee128c2fcfc                                           4   
n0048ac83aff7194                                           1   
n0055a2401ba6480                                     

In [6]:
target_cols = [c for c in train.columns if c.startswith("target_")]
print(target_cols)

['target_agnes_20', 'target_agnes_60', 'target_alpha_20', 'target_alpha_60', 'target_bravo_20', 'target_bravo_60', 'target_caroline_20', 'target_caroline_60', 'target_charlie_20', 'target_charlie_60', 'target_claudia_20', 'target_claudia_60', 'target_cyrusd_20', 'target_cyrusd_60', 'target_delta_20', 'target_delta_60', 'target_echo_20', 'target_echo_60', 'target_ender_20', 'target_ender_60', 'target_jasper_20', 'target_jasper_60', 'target_jeremy_20', 'target_jeremy_60', 'target_ralph_20', 'target_ralph_60', 'target_rowan_20', 'target_rowan_60', 'target_sam_20', 'target_sam_60', 'target_teager2b_20', 'target_teager2b_60', 'target_tyler_20', 'target_tyler_60', 'target_victor_20', 'target_victor_60', 'target_waldo_20', 'target_waldo_60', 'target_xerxes_20', 'target_xerxes_60']


In [7]:
import pandas as pd
import lightgbm as lgb
import json
import numpy as np

# 피처/타겟 분리
feature_cols = [c for c in train.columns if c.startswith("feature_")]
target_col = "target_cyrusd_20"  # 메인 타겟

# 메모리 절약: era 일부만 샘플링
eras = train["era"].unique()
sampled_eras = eras[:200]  # 전체 중 앞 200개 era만 사용
sample = train[train["era"].isin(sampled_eras)]

X = sample[feature_cols]
y = sample[target_col]

# LightGBM으로 feature importance 계산
model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# 중요도 정렬
importance = pd.Series(model.feature_importances_, index=feature_cols)
importance = importance.sort_values(ascending=False)

# 상위 50% 피처만 선택
threshold = importance.median()
selected_features = importance[importance >= threshold].index.tolist()

print(f"전체 피처 수: {len(feature_cols)}")
print(f"선택된 피처 수: {len(selected_features)}")
print(importance.head(20))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.873251 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12015
[LightGBM] [Info] Number of data points in the train set: 833533, number of used features: 2403
[LightGBM] [Info] Start training from score 0.500059
전체 피처 수: 2748
선택된 피처 수: 1517
feature_saharan_parietal_catch                   12
feature_scenic_unartistic_tache                  10
feature_delusory_fake_lower                      10
feature_probative_jolliest_february              10
feature_overstrong_burrier_guevara                9
feature_supplicatory_ski_broach                   9
feature_threadbare_reviving_franco                9
feature_droll_clumsy_gleaner                      9
feature_inartistic_necrological_showgirl          8
feature_catechetical_paragogical_accouterment     8
feature_roilier_slippered_patti          

In [8]:
# 선택된 피처 목록 저장
import json

with open("selected_features.json", "w") as f:
    json.dump(selected_features, f)

print(f"저장 완료! 선택된 피처 수: {len(selected_features)}")

저장 완료! 선택된 피처 수: 1517


In [1]:
import pandas as pd
import numpy as np
import json
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

In [2]:
train = pd.read_parquet("train.parquet")
validation = pd.read_parquet("validation.parquet")

with open("selected_features.json", "r") as f:
    selected_features = json.load(f)

print(f"선택된 피처 수: {len(selected_features)}")

선택된 피처 수: 1517


In [3]:
eras = train["era"].unique()
train_sample = train[train["era"].isin(eras[:50])]
top_500_features = selected_features[:500]

scaler = StandardScaler()
pca = PCA(n_components=100)

X_train = train_sample[top_500_features]
X_train_scaled = scaler.fit_transform(X_train)
X_train_pca = pca.fit_transform(X_train_scaled)

# validation도 샘플링
val_eras = validation["era"].unique()
val_sample = validation[validation["era"].isin(val_eras[:50])]

X_val = val_sample[top_500_features]
X_val_scaled = scaler.transform(X_val)
X_val_pca = pca.transform(X_val_scaled)

print(f"학습 데이터 shape: {X_train_pca.shape}")
print(f"검증 데이터 shape: {X_val_pca.shape}")

학습 데이터 shape: (150323, 100)
검증 데이터 shape: (290312, 100)


In [4]:
target_col = "target_cyrusd_20"

y_train = train_sample[target_col]
y_val = val_sample[target_col]

In [5]:
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=5,
    num_leaves=32,
    random_state=42
)

model.fit(X_train_pca, y_train)
print("학습 완료!")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035866 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25500
[LightGBM] [Info] Number of data points in the train set: 150323, number of used features: 100
[LightGBM] [Info] Start training from score 0.500105
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [6]:
val_preds = model.predict(X_val_pca)
corr, _ = spearmanr(val_preds, y_val)
print(f"Validation CORR: {corr:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Validation CORR: 0.0127
